In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.util import img_as_uint

import os
import tifffile
import pickle

from skimage.transform import estimate_transform, warp
from tqdm import tqdm

# Registration

## calculate transformation

In [31]:
reference_dir = r'Y:\coskun-lab\Zhou\12_MSG\20240604_msg\00_registered\v2\158_001\reference'
transform_dir = r'Y:\coskun-lab\Zhou\12_MSG\20240604_msg\00_registered\v2\158_001\transformation'

In [32]:
cycles = os.listdir(reference_dir)
cycles.sort()
# cycles = cycles[:]

In [33]:
station_reference = pd.read_csv(os.path.join(reference_dir, cycles[0]), index_col=0)

In [34]:
for cycle in tqdm(cycles[1:]):
    moving_reference = pd.read_csv(os.path.join(reference_dir, cycle), index_col=0)
    tform = estimate_transform('affine',
                               np.stack((moving_reference['col'].tolist(), moving_reference['row'].tolist()), axis=1),
                               np.stack((station_reference['col'].tolist(), station_reference['row'].tolist()),axis=1))
    with open(os.path.join(transform_dir, cycle.split('.')[0] + '.pkl'), 'wb') as f:
        pickle.dump(tform, f)

100%|██████████| 3/3 [00:00<00:00, 23.05it/s]


## shifting

In [2]:
from joblib import Parallel, delayed

In [88]:
raw_dir = r'Y:\coskun-lab\Zhou\12_MSG\20230926_msg\00_registered\v2\134_003\raw'
registered_dir = r'Y:\coskun-lab\Zhou\12_MSG\20230926_msg\00_registered\v2\134_003\registered'
transform_dir = r'Y:\coskun-lab\Zhou\12_MSG\20230926_msg\00_registered\v2\134_003\transformation'
cycles = os.listdir(transform_dir)

In [89]:
channels = ['C1','C2','C3','C4']
for c in channels:
    os.makedirs(os.path.join(registered_dir, c), exist_ok=True)

In [90]:
def transform_image(ch, cycle, shape, im_l):
    cy = cycle.split('.')[0]
    tform = pd.read_pickle(os.path.join(transform_dir, cy + '.pkl'))
    file_name = next(filter(lambda file: '_'+cy+'_' in file, im_l), None)
    img = tifffile.imread(os.path.join(raw_dir, ch, file_name))
    warped_image = warp(img, tform.inverse, output_shape=shape)
    tifffile.imwrite(os.path.join(registered_dir, ch, file_name), img_as_uint(warped_image))

In [91]:
fixed_im = tifffile.imread(r"Y:\coskun-lab\Zhou\12_MSG\20230926_msg\00_registered\v2\134_003\raw\C1\C1_01_msg_134_ptprc_hla-dra_gapdh_003.tif")
fixed_shape = fixed_im.shape
for ch in channels:
    im_l = os.listdir(os.path.join(raw_dir, ch))
    _ = Parallel(n_jobs=4, backend='threading',verbose=1)(delayed(transform_image)(ch, cycle, fixed_shape, im_l) for cycle in cycles)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:   25.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:   25.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:   26.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:   27.4s finished


# Cropping if necessary

In [108]:
im_dir = r'Y:\coskun-lab\Zhou\12_MSG\20230926_msg\00_registered\v2\103_004\registered'
cropping_dir = r'Y:\coskun-lab\Zhou\12_MSG\20230926_msg\00_registered\v2\103_004\cropping'

In [109]:
for channels in ['C1','C2','C3','C4']:
    os.makedirs(os.path.join(cropping_dir, channels), exist_ok=True)

In [110]:
cropping = pd.read_csv(os.path.join(cropping_dir, 'cropping.csv'), index_col=0)
cropping = cropping.astype(int)

In [111]:
for ch in ['C1','C2','C3','C4']:
    im_l = os.listdir(os.path.join(im_dir, ch))
    for file in tqdm(im_l):
        if not file.endswith('.tif'):
            continue
        img = tifffile.imread(os.path.join(im_dir, ch, file))
        cropped_img = img[cropping['row'].min():cropping['row'].max(), cropping['col'].min():cropping['col'].max()]
        tifffile.imwrite(os.path.join(cropping_dir, ch, file), img_as_uint(cropped_img))

100%|██████████| 6/6 [00:10<00:00,  1.78s/it]


# dot detection

In [25]:
from skimage.filters import threshold_triangle
from skimage.feature import peak_local_max
from skimage.util import img_as_uint, img_as_float

In [26]:
def detect_dots(img):
    thre_abs = threshold_triangle(img)*1.5
    return peak_local_max(img, min_distance=3,threshold_abs = thre_abs)

def filter_dots(dots, background):
    thre_background = threshold_triangle(background) * 2.5
    return dots[background[dots[:, 0], dots[:, 1]] < thre_background]

In [132]:
dots_dir = r'Y:\coskun-lab\Zhou\12_MSG\20240604_msg\00_dots'

In [168]:
registered_im_dir = r'Y:\coskun-lab\Zhou\12_MSG\20240604_msg\00_registered\v2\158_001\cropping'
channels = ['C1', 'C2', 'C3', 'C4']
cycles = ['03','04','06','07']

In [169]:
ch = 'C1'
channel_dir = os.path.join(registered_im_dir, ch)
channel_fn_dict = {}
for im in os.listdir(channel_dir):
    if im.endswith('.tif'):
        cycle = im.split('_')[1]
        channel_fn_dict[cycle] = im[3:]

In [170]:
def detect_dots(img):
    thre_abs = threshold_triangle(img)*1.5
    return peak_local_max(img, min_distance=3,threshold_abs = thre_abs)

def filter_dots(dots, background):
    thre_background = threshold_triangle(background) * 2.5
    return dots[background[dots[:, 0], dots[:, 1]] < thre_background]

In [172]:
detected_dots = []
for cycle in tqdm(cycles):

    ch2 = tifffile.imread(os.path.join(registered_im_dir, 'C2', 'C2_'+channel_fn_dict[cycle]))
    ch2 = img_as_float(ch2)
    ch3 = tifffile.imread(os.path.join(registered_im_dir, 'C3', 'C3_'+channel_fn_dict[cycle]))
    ch3 = img_as_float(ch3)
    ch4 = tifffile.imread(os.path.join(registered_im_dir, 'C4', 'C4_'+channel_fn_dict[cycle]))
    ch4 = img_as_float(ch4)
    print('Images loaded')

    # ch2_dots = detect_dots(ch2)
    # print('C2 detected')
    ch3_dots = detect_dots(ch3)
    print('C3 detected')
    ch4_dots = detect_dots(ch4)
    print('C4 detected')

    # ch2_dot_filter = filter_dots(ch2_dots, ch3)
    ch3_dot_filter = filter_dots(ch3_dots, ch2)
    ch4_dot_filter = filter_dots(ch4_dots, ch3)
    # ch2_df = pd.DataFrame(ch2_dot_filter, columns=['row', 'col'])
    ch3_df = pd.DataFrame(ch3_dot_filter, columns=['row', 'col'])
    ch4_df = pd.DataFrame(ch4_dot_filter, columns=['row', 'col'])
    detected_dots.append([ch3_df, ch4_df])

  0%|          | 0/4 [00:00<?, ?it/s]

Images loaded
C3 detected
C4 detected


 25%|██▌       | 1/4 [01:24<04:12, 84.21s/it]

Images loaded
C3 detected
C4 detected


 50%|█████     | 2/4 [01:49<01:39, 49.74s/it]

Images loaded
C3 detected
C4 detected


 75%|███████▌  | 3/4 [02:12<00:37, 37.51s/it]

Images loaded
C3 detected
C4 detected


100%|██████████| 4/4 [02:35<00:00, 38.87s/it]


In [173]:
gens = [['GAPDH','CXCL11'],
        ['LYZ','CXCL10'],
        #['EMPTY','CXCL13','CXCL12'],
        ['TNFAIP3','TNFRSF4'],
        ['CXCR3','CXCR5']]

In [174]:
dots_l = []
for i in range(len(detected_dots)):
    for j in range(len(detected_dots[i])):
        df = detected_dots[i][j]
        df['gene'] = gens[i][j]
        dots_l.append(df)

In [175]:
dots_df = pd.concat(dots_l, ignore_index=True)
dots_df = dots_df[dots_df['gene'] != 'EMPTY']

In [176]:
dots_df.to_csv(os.path.join(dots_dir, '158.csv'))

# count dots

In [177]:
from scipy.ndimage import distance_transform_edt, label
from skimage.segmentation import expand_labels
from skimage.measure import regionprops_table

In [178]:
analysis_dir = r'Y:\coskun-lab\Zhou\12_MSG\20240604_msg\00_analysis'

In [179]:
mask = tifffile.imread(r"Y:\coskun-lab\Zhou\12_MSG\20240604_msg\00_registered\v2\158_001\mask\C1_03_msg_158_empty_gapdh_cxcl11_001_cp_masks.tif")
detected_dots_df = pd.read_csv(os.path.join(dots_dir, '158.csv'), index_col=0)

In [180]:
expanded = expand_labels(mask, distance=5)

In [181]:
detected_dots_df['label'] = expanded[detected_dots_df['row'].astype(int), detected_dots_df['col'].astype(int)]
detected_dots_df_filtered = detected_dots_df[detected_dots_df['label'] > 0]
detected_dots_df_filtered.to_csv(os.path.join(analysis_dir,'filtered_doots_158.csv'))

In [182]:
genes = detected_dots_df['gene'].unique().tolist()
gene_count = {}
for gene in genes:
    gene_df = detected_dots_df_filtered[detected_dots_df_filtered['gene'] == gene]
    gene_count[gene] = gene_df['label'].value_counts()

In [183]:
gene_count_df = pd.DataFrame(gene_count)
gene_count_df = gene_count_df.fillna(0)

In [184]:
gene_count_df

,GAPDH,CXCL11,LYZ,CXCL10,TNFRSF4,CXCR3,CXCR5
label,,,,,,,
2,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,0.0,11.0,0.0,4.0,1.0,0.0,1.0
7,0.0,1.0,0.0,0.0,0.0,0.0,0.0
8,0.0,1.0,0.0,1.0,0.0,0.0,0.0
17,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...
13450,0.0,12.0,0.0,0.0,0.0,0.0,0.0
13451,0.0,13.0,0.0,0.0,0.0,0.0,0.0
13452,0.0,12.0,0.0,0.0,0.0,0.0,0.0


In [185]:
gene_count_df.to_csv(os.path.join(analysis_dir, 'gene_count_158.csv'))

In [186]:
cells_properties = regionprops_table(mask, properties=['label', 'area', 'centroid'])
cells_properties_df = pd.DataFrame(cells_properties)

In [187]:
cells_properties_df.index = cells_properties_df['label']

In [188]:
cells_properties_df = cells_properties_df.merge(gene_count_df, left_index=True, right_index=True)
cells_properties_df = cells_properties_df.fillna(0)

In [189]:
cells_properties_df.to_csv(os.path.join(analysis_dir, 'cell_properties_158.csv'))